In [80]:
import torch
import torch.nn as nn
import numpy as np
from torch import Tensor
from matplotlib import pyplot as plt
from sklearn.datasets import make_moons, make_circles

In [ ]:
class Flow(nn.Module): 
    '''
    Defines the architecture for flow matching
    '''

    def __init__(self, dim: int = 2, h: int = 64, cond: bool = False): # fully connected neural network for learning velocity field 
        super().__init__()
        
        self.cond = cond 
        if self.cond: 
            print("Conditioning model on xc")
            n = 2
        else: 
            print("Neglect condition")
            n = 1
            
        self.net = nn.Sequential(
            nn.Linear(n * dim + 1, h),  
            nn.ELU(),              
            nn.Linear(h, h),       
            nn.ELU(),             
            nn.Linear(h, h),      
            nn.ELU(),              
            nn.Linear(h, dim)       
        )
    
    def forward(self, t: Tensor, xt: Tensor, xc: Tensor) -> Tensor:
        '''
        Concatenates time and position vectors for forward pass through neural network  
        '''

        if self.cond: 
            combined_input = torch.cat((t, xc, xt), -1)
        else:
            combined_input = torch.cat((t, xt), -1)
            
        return self.net(combined_input)
    
    def step(self, xt: Tensor, xc: Tensor, t_start: Tensor, t_end: Tensor) -> Tensor: 
        '''
        Performs midpoint integration to sample transform x0 to x1 (i.e., sample from p1, starting from p0)
        '''
      
        t_start = t_start.view(1, 1).expand(xt.shape[0], 1)
        midpoint_time = t_start + (t_end - t_start) / 2 # evalute the midpoint 
        if self.cond:
            intermediate_value = self(xt=xt, t=t_start, xc=xc) # compute the velocity at this point using the trained neural network 
        else: 
            intermediate_value = self(xt=xt, t=t_start, xc=xc) # compute the velocity at this point using the trained neural network
        updated_xt = xt + intermediate_value * (t_end - t_start) / 2 # move xt using computed velocity 
        midpoint_output = self(t=midpoint_time, xt=updated_xt, xc=xc) # compute vt again 
        
        return xt + (t_end - t_start) * midpoint_output # compute xt

In [61]:
# Training loop
def train():
    '''
    Defines the training procedure for the neural network
    '''
    for _ in range(1000000): #10000000
        
        if _ == 0: 
            x1 = Tensor(make_moons(1, noise=0.05)[0]) #Tensor([1, 2, 3, 4, 5, 6]).unsqueeze(0) #
            print(x1)
        x0 = torch.randn_like(x1)
        xc = x1  #Tensor([1, 0, 3, 0, 5, 0]).unsqueeze(0)
        t = torch.rand(len(x1), 1) #torch.rand(1).unsqueeze(0) #

        xt = (1 - t) * x0 + t * x1

        # Compute loss 
        loss = loss_fn(flow(t=t, xt=xt, xc=xc), x1 - x0)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        print(f"Loss at epoch {_}: {loss.item()}")

        if loss.item() < 0.000000001:
            break 

# Initialize model and optimizer
flow = Flow()
optimizer = torch.optim.Adam(flow.parameters(), 1e-3)
loss_fn = nn.MSELoss()
train()


Neglect condition
tensor([[0.0883, 0.4923]])
Loss at epoch 0: 0.5523227453231812
Loss at epoch 1: 1.9424875974655151
Loss at epoch 2: 1.03721284866333
Loss at epoch 3: 1.9826250076293945
Loss at epoch 4: 0.5941504836082458
Loss at epoch 5: 0.1066085547208786
Loss at epoch 6: 0.732094407081604
Loss at epoch 7: 0.6733213067054749
Loss at epoch 8: 1.5057249069213867
Loss at epoch 9: 0.2032475769519806
Loss at epoch 10: 0.11161908507347107
Loss at epoch 11: 0.3504013419151306
Loss at epoch 12: 0.8670490980148315
Loss at epoch 13: 2.030046224594116
Loss at epoch 14: 3.411677122116089
Loss at epoch 15: 0.3473949432373047
Loss at epoch 16: 0.5565155744552612
Loss at epoch 17: 0.5136981010437012
Loss at epoch 18: 0.033642157912254333
Loss at epoch 19: 1.0680842399597168
Loss at epoch 20: 0.1911093294620514
Loss at epoch 21: 0.44642969965934753
Loss at epoch 22: 0.3202872574329376
Loss at epoch 23: 0.47550439834594727
Loss at epoch 24: 2.1284961700439453
Loss at epoch 25: 1.4815301895141602
Los

In [101]:
flow.eval()
x = []
for n in range(1000):
    x1 = torch.Tensor([[0.0883, 0.4923]])
    x0 = x0 = torch.randn_like(x1)
    g = flow.step(xt=x0, xc=x1, t_start=torch.Tensor([0]), t_end=torch.Tensor([1]))
    g = g.detach().numpy().squeeze()
    x.append(g)
print(x)
plt.plot(n,x[0,0])

[array([0.08823818, 0.45158476], dtype=float32), array([0.08982663, 0.45652437], dtype=float32), array([0.05548489, 0.47710255], dtype=float32), array([0.04475212, 0.49354434], dtype=float32), array([0.0956257, 0.5017381], dtype=float32), array([0.03849792, 0.49142796], dtype=float32), array([0.00498426, 0.49170417], dtype=float32), array([0.00095457, 0.5776582 ], dtype=float32), array([-0.01043403,  0.52349734], dtype=float32), array([0.06929338, 0.4624182 ], dtype=float32), array([0.01323688, 0.48197472], dtype=float32), array([0.05446875, 0.48322758], dtype=float32), array([0.11256486, 0.46737254], dtype=float32), array([0.06745875, 0.5124928 ], dtype=float32), array([0.07109183, 0.5014637 ], dtype=float32), array([0.11204912, 0.49563375], dtype=float32), array([0.1027458 , 0.55150515], dtype=float32), array([0.08463299, 0.47462747], dtype=float32), array([0.10432762, 0.47021648], dtype=float32), array([-0.03540111,  0.5329778 ], dtype=float32), array([0.08253342, 0.5361895 ], dtype

TypeError: list indices must be integers or slices, not tuple